In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.model_selection import GridSearchCV

In [16]:
# loading data

df = pd.read_csv('insurance.csv')

In [17]:
# preprocessing

m = {'yes' : 1, 'no' : 0}
df['smoker'] = df['smoker'].map(m)

m = {'male' : 1, 'female' : 0}
df.rename(columns={'sex' : 'isMale'}, inplace=True)
df['isMale'] = df['isMale'].map(m)

df = pd.get_dummies(df, columns=['region'], drop_first=True, dtype='int')

In [18]:
# x and y split

X = df.drop(columns='charges')
y = df['charges']

In [19]:
# train test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
# scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns = X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns = X_test.columns)

In [21]:
# score error function

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def score_error(model_name, y_test, y_pred):
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"{model_name} Model Performance\n")
    print(f"R Squared (R2) : {r2:.4f}")
    print(f"Mean Absolute Error (MAE) : {mae:.2f}")
    print(f"Root Mean Squared Error (RMSE) : {rmse:.2f}")

In [22]:
# linear regression

from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
score_error('Linear Regression', y_test, lr_pred)

Linear Regression Model Performance

R Squared (R2) : 0.7836
Mean Absolute Error (MAE) : 4181.19
Root Mean Squared Error (RMSE) : 5796.28


In [23]:
# sgd regressor

from sklearn.linear_model import SGDRegressor

sgd_param_grid = {
    'penalty': ['l2', 'l1', 'elasticnet'],
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'learning_rate': ['constant', 'optimal', 'invscaling', 'adaptive'],
    'eta0': [0.01, 0.1]
}

sgd_grid = GridSearchCV(
    estimator=SGDRegressor(random_state=42, max_iter=100000),
    param_grid=sgd_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

sgd_grid.fit(X_train_scaled, y_train)

best_sgd = sgd_grid.best_estimator_
sgd_pred = best_sgd.predict(X_test_scaled)

print(f"Best SGD Regressor Parameters : {sgd_grid.best_params_}\n")
score_error('SGD Regressor', y_test, sgd_pred)

Best SGD Regressor Parameters : {'alpha': 0.01, 'eta0': 0.01, 'learning_rate': 'invscaling', 'penalty': 'l2'}

SGD Regressor Model Performance

R Squared (R2) : 0.7831
Mean Absolute Error (MAE) : 4194.52
Root Mean Squared Error (RMSE) : 5803.01


In [24]:
# decision tree regressor

from sklearn.tree import DecisionTreeRegressor

dt_param_grid = {
    'max_depth': [None, 3, 5, 7, 10],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10]
}

dt_grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=dt_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

best_dt = dt_grid.best_estimator_
dt_pred = best_dt.predict(X_test)

print(f"Best Decision Tree Parameters: {dt_grid.best_params_}\n")
score_error('Decision Tree Regressor', y_test, dt_pred)

Best Decision Tree Parameters: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}

Decision Tree Regressor Model Performance

R Squared (R2) : 0.8531
Mean Absolute Error (MAE) : 2865.64
Root Mean Squared Error (RMSE) : 4776.26


In [25]:
# random forest regressor

from sklearn.ensemble import RandomForestRegressor

rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [12, 14, 16, 18],
}

rf_grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=rf_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
rf_pred = best_rf.predict(X_test)

print(f"Best Random Forest Parameters: {rf_grid.best_params_}\n")
score_error('Random Forest', y_test, rf_pred)

Best Random Forest Parameters: {'max_depth': 5, 'min_samples_leaf': 16, 'min_samples_split': 2, 'n_estimators': 50}

Random Forest Model Performance

R Squared (R2) : 0.8747
Mean Absolute Error (MAE) : 2548.26
Root Mean Squared Error (RMSE) : 4411.35


In [26]:
# adaboost regressor

from sklearn.ensemble import AdaBoostRegressor

ada_param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
    'loss': ['linear', 'square', 'exponential']
}

ada_grid = GridSearchCV(
    estimator=AdaBoostRegressor(random_state=42),
    param_grid=ada_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

ada_grid.fit(X_train, y_train)

best_ada = ada_grid.best_estimator_
ada_pred = best_ada.predict(X_test)

print(f"Best AdaBoost Parameters: {ada_grid.best_params_}\n")
score_error('AdaBoost', y_test, ada_pred)

Best AdaBoost Parameters: {'learning_rate': 0.01, 'loss': 'linear', 'n_estimators': 50}

AdaBoost Model Performance

R Squared (R2) : 0.8581
Mean Absolute Error (MAE) : 3089.72
Root Mean Squared Error (RMSE) : 4693.24



As we can see here Random Forest has best R2 Score and least error.

R2 Score of different models :-
    LR : 0.7836,
    SGD : 0.7831,
    Decision Tree : 0.8531,
    Random Forest : 0.8747,
    AdaBoost : 0.8581

Linear models like LR and SGD have low R2 as compared to others as data is not linear.

Here RF outperformed as here it is capturing non linear nicely and also preventing overfitting that can happen in case of single Decision Tree.
